- TODO: continue video and add experiments
- TODO: how does kaiming init gain help if its only for first step
- TODO: review TODOs
- TODO: review all code
- TODO: prediction vs conclusion

- TODO: batch norm
- TODO: output_layer_weight_scale
- TODO: bug saturation is being calcualted with no nonlinearity
- TODO: gradient std uses learning rate?
- TODO: fix printing of mean and std
- TODO: add final run, test running batch norm before and after activations, test adding or not adding to output layer
- TODO: ask gpt 4o to analyze the results of training runs and summarize the patterns that were found to draw conclusions
call plot_model_graphs run_experiment
TODO: learning rate is not always increased

# Optimizing Neural Networks: Initializations, Activations, and Gradient Flow - Part 2

This notebook is a reconstruction of Andrej Karpathy's [Building makemore Part 3: Activations & Gradients, BatchNorm](https://www.youtube.com/watch?v=P6sfmUTpUmc). It covers:
- Initializing Neural Networks effectively for better training.
- Analyzing gradient flows to identify learning bottlenecks.
- Using [Batch Normalization](https://en.wikipedia.org/wiki/Batch_normalization) to stabilize gradients during training.

The notebook was broken into two parts because the lesson was a bit too long.
The first part is here: TODO and it introduced the topics above. In this one we'll focus on starting to migrate the code to pytorch, and then perform multiple experiments with initialization and batch norm to truly understand how they affect data and gradient flow through the network to gain a more intutivie understanding.

*The purpose of this notebook is for my own self-learning, and shouldn't add much to the original lesson beyond extra verbosity. Parts of the notebook, namely the code snippets, may have been copied from the source lesson verbatim.*

## Setup


In [ ]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

import torch
import random

# Check if CUDA is available and set the device
DEVICE = torch.device("cpu")#'cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
BLOCK_SIZE = 3

words = open("names.txt", "r").read().splitlines()
len(words), words[:8]

# Create list with all unique characters found in the raw data (sorted alphabetically)
chars = sorted(list(set(''.join(words))))

# Create lookup table for converting each possible character to a unique integer
stoi_map = {s:i+1 for i,s in enumerate(chars)}
EOS_CHAR = "." # The character used to represent the end of a word
EOS_CHAR_INDEX = 0
stoi_map[EOS_CHAR] = EOS_CHAR_INDEX # Assign index zero to EOS char

# Create reverse lookup table
itos_map = {i:s for s,i in stoi_map.items()}
itos_map

# Count the size of the vocabulary
# (how many different characters exist in the dataset)
vocabulary_size = len(stoi_map)

def build_dataset(words, block_size=BLOCK_SIZE, verbose=False):
  X, Y = [], [] # Initialize the dataset
  for word in words:
    if verbose: print(word)
    context = [0] * block_size # Initialize the context with EOS characters
    for char in word + EOS_CHAR: # For each character in the current word (plus the EOS stop character)
      X.append(context) # Add the context accumulated so far as the input
      char_i = stoi_map[char] # Convert the current character to an integer
      Y.append(char_i) # Add the current character as the output (previous context must predict current character)
      if verbose: print(f"{''.join([itos_map[i] for i in context])} => {char}")
      context = context[1:] + [char_i] # Update the context by popping out the oldest character and pushing in the current one (FIFO buffer)

  # Convert the dataset to tensors and return them
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

def build_split_dataset(words, block_size=BLOCK_SIZE):
  # Shuffle the words so that the data splits are random
  words = words.copy() # Copy so we don't modify the original reference when shuffling
  random.shuffle(words)

  # Split the dataset
  n1 = int(0.8 * len(words)) # First data split point
  n2 = int(0.9 * len(words)) # Second data split point
  training_set = build_dataset(words[:n1], block_size=block_size) # Create training set from range [0%-80%]
  validation_set = build_dataset(words[n1:n2], block_size=block_size) # Create validation set from range [80%-90%]
  test_set = build_dataset(words[n2:], block_size=block_size) # Create test set from range [90%-100%]

  # Return the splits
  return training_set, validation_set, test_set

# Create the dataset and split it into training, validation and test sets
training_set, validation_set, test_set = build_split_dataset(words)

# Output the number of examples in each dataset split
n_training_set = len(training_set[0])
n_validation_set = len(validation_set[0])
n_test_set = len(test_set[0])
n_total = n_training_set + n_validation_set + n_test_set
{
    "n_total" : n_total,
    "n_training" : n_training_set,
    "n_training_percentage" : round(n_training_set / n_total * 100),
    "n_validation" : n_validation_set,
    "n_validation_percentage" : round(n_validation_set / n_total * 100),
    "n_test" : n_test_set,
    "n_test_percentage" : round(n_test_set / n_total * 100)
}

## PyTorchify 🐍🔥

We need to add the code from Part 1 to continue the lesson, so let's use the opportunity to re-create it using PyTorch instead. We're going to be implementing primitives PyTorch already provides, like `Linear` and `BatchNorm1d`, but it's important to do so as a stepping stone to understand what happens inside these objects. Ultimately, we'll end up using PyTorch directly and not reinvent the wheel, but one thing at a time.

Let's start with creating a custom layer that mimics the functionality of [torch.nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html):

In [ ]:
# This is a linear layer: it performs matmul
# between input and weights, then adds a bias
class Linear:

  def __init__(self, fan_in, fan_out, bias=True, kaiming_init=True, generator=None):
    # Create layer weights with shape (fan_in, fan_out),
    # initialized using simplified kaiming init (divide by square root of fan-in)
    kaiming_scale = fan_in**0.5 if kaiming_init else 1.0
    self.weight = torch.randn((fan_in, fan_out), device=DEVICE, generator=generator) / kaiming_scale

    # Create layer biases with shape (fan_out)
    self.bias = torch.zeros(fan_out, device=DEVICE) if bias else None

  def __call__(self, x):
    self.out = x @ self.weight # matmul between input and weights
    if self.bias is not None: self.out += self.bias # Add bias
    return self.out # Return pre-activation

  def parameters(self):
    # Return a list with all the parameters in the layer
    return [self.weight] + ([] if self.bias is None else [self.bias])

# Test the layer
Linear(1, 2)(torch.randn((2, 1), device=DEVICE))

Next, let's create a custom batch normalization layer that mimics the functionality of [torch.nn.BatchNorm1d](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html):

In [ ]:
class BatchNorm1d:

  def __init__(
      self,
      dim,
      eps=1e-5,
      momentum=0.1
    ):
    # Small value to add during normalization
    # in order to avoid division by zero
    self.eps = eps

    # The momentum at which the current batch's statistics should
    # affect the running average statistics being kept by this
    # layer to be used later during inference
    self.momentum = momentum

    # Flag indicating if the model is being used for training or inference
    # (this layer will behave differently in each case, namely in the way it
    # retrieves the statistics to be used to normalize the inputs that go through it)
    self.training = True

    # The gain used to scale to the normalized activations (learned parameter)
    self.gamma = torch.ones(dim, device=DEVICE)

    # The bias to shift the normalized activations (learned parameter)
    self.beta = torch.zeros(dim, device=DEVICE)

    # Buffers userd to keep a running averages
    # of the batch statistics (to be used during inference)
    self.running_mean = torch.zeros(dim, device=DEVICE)
    self.running_variance = torch.ones(dim, device=DEVICE)

  def __call__(self, x):
    # In case the model is training calculate
    # the batch's mean and variance
    if self.training:
      x_mean = x.mean(0, keepdim=True)
      x_variance = x.var(0, keepdim=True)
    # Otherwise use the running statistics
    # that were calculated during training
    else:
      x_mean = self.running_mean
      x_variance = self.running_variance

    # Normalize the input by subtracting the
    # mean and dividing by the variance
    x_normalized = (x - x_mean) / torch.sqrt(x_variance + self.eps)

    # Scale and shift the normalized input and set it as the output
    self.out = self.gamma * x_normalized + self.beta

    # In case the model is training update the running
    # average statistics with the current batch statistics
    if self.training:
      # Don't track gradients during the following operations
      with torch.no_grad():
        # Add a bit of the current batch's mean to the running average of the mean
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * x_mean

        # Add a bit of the current batch's variance to the running average of the mean
        self.running_variance = (1 - self.momentum) * self.running_variance + self.momentum * x_variance

    # Return the normalized activations
    return self.out

  def parameters(self):
    # Return the layer's parameters (batch normalization needs to
    # learn how to scale and shift the normalized activations)
    return [self.gamma, self.beta]

# TODO: what does dim do?
# Test the layer
BatchNorm1d(1)(torch.randn((2, 1), device=DEVICE))

Finally, let's implement a $tanh$ activation function that mimics [torch.nn.Tanh](https://pytorch.org/docs/stable/generated/torch.nn.Tanh.html):

In [ ]:
# This is a Tanh activation layer
# (e.g., to add after a linear layer)
class Tanh:

  def __call__(self, x):
    # Apply tanh to input and return result
    self.out = torch.tanh(x)
    return self.out

  def parameters(self):
    # This layer has no parameters
    return []

# Test the layer
Tanh()(torch.tensor([[-0.1, 0.1], [-3, 3], [-100, 100]], device=DEVICE))

Now that we've created all the necessary layers, let's recreate the `build_model()` method using these custom constructs. We'll also include some extra components for experiments later in the notebook:

In [ ]:
def build_model(
    vocabulary_size,
    block_size=BLOCK_SIZE,          # Number of previous characters used to predict the next character
    embedding_size=10,              # Size of the dense vector used to represent each character
    n_hidden=100,                   # Number of neurons in the hidden layer
    non_linearity=Tanh,
    batch_norm=False,
    output_layer_weight_scale=0.1,
    kaiming_init_gain=5/3,
    seed=SEED                       # The RNG seed to use when initializing parameters (for reproducibility)
):
  if batch_norm is True: batch_norm = "preactivation"

  # Initialize the generator with the provided seed
  generator = torch.Generator(DEVICE).manual_seed(seed)

  # TODO: why no generator in linear layers?
  C = torch.randn((vocabulary_size, embedding_size), generator=generator, device=DEVICE)

  kaiming_init = True if kaiming_init_gain is not None else False

  layers = [
    # Linear -> BatchNorm1d -> Tanh
    Linear(embedding_size * block_size, n_hidden, kaiming_init=kaiming_init, generator=generator),  #TODO: mirror api for kaiming init
    BatchNorm1d(n_hidden) if batch_norm == "preactivation" else None,
    non_linearity() if non_linearity else None,
    BatchNorm1d(n_hidden) if batch_norm == "activation" else None,

    # Linear -> BatchNorm1d -> Tanh
    Linear(n_hidden, n_hidden, kaiming_init=kaiming_init, generator=generator),
    BatchNorm1d(n_hidden) if batch_norm == "preactivation" else None,
    non_linearity() if non_linearity else None,
    BatchNorm1d(n_hidden) if batch_norm == "activation" else None,

    # Linear -> BatchNorm1d -> Tanh
    Linear(n_hidden, n_hidden, kaiming_init=kaiming_init, generator=generator),
    BatchNorm1d(n_hidden) if batch_norm == "preactivation" else None,
    non_linearity() if non_linearity else None,
    BatchNorm1d(n_hidden) if batch_norm == "activation" else None,

    # Linear -> BatchNorm1d -> Tanh
    Linear(n_hidden, n_hidden, kaiming_init=kaiming_init, generator=generator),
    BatchNorm1d(n_hidden) if batch_norm == "preactivation" else None,
    non_linearity() if non_linearity else None,
    BatchNorm1d(n_hidden) if batch_norm == "activation" else None,

    # Linear -> BatchNorm1d -> Tanh
    Linear(n_hidden, n_hidden, kaiming_init=kaiming_init, generator=generator),
    BatchNorm1d(n_hidden) if batch_norm == "preactivation" else None,
    non_linearity() if non_linearity else None,
    BatchNorm1d(n_hidden) if batch_norm == "activation" else None,

    # Linear -> BatchNorm1d
    Linear(n_hidden, vocabulary_size, kaiming_init=kaiming_init, generator=generator),
    BatchNorm1d(vocabulary_size) if batch_norm == "preactivation" else None
  ]
  layers = [layer for layer in layers if layer]

  # Don't track gradients for code within this block
  with torch.no_grad():
    # Make last layer less confident
    if batch_norm: layers[-1].gamma *= output_layer_weight_scale # TODO: what is going on here
    else: layers[-1].weight *= output_layer_weight_scale

    # Use tanh gain in all layers except the last one
    if kaiming_init:
      for layer in layers[:-1]:
        if isinstance(layer, Linear): layer.weight *= kaiming_init_gain

  # Enable gradient tracking for all model parameters
  parameters = [C] + [p for layer in layers for p in layer.parameters()]
  for p in parameters: p.requires_grad = True

  return layers, parameters

model = build_model(vocabulary_size)
layers, parameters = model
len(layers), sum(p.nelement() for p in parameters)

Now, let's recreate the `train()` function:

In [ ]:
import torch.nn.functional as F

# The default learning rate
LEARNING_RATE = 0.1

def train(
    dataset,           # The dataset to train on (a tuple with a list of inputs and a list of respective outputs)
    model,             # The model parameters
    n_steps,           # Train for N steps
    learning_rate=LEARNING_RATE, # The scale at which gradients should be applied to the parameters at each step
    batch_size=32,     # The size of the mini-batch to randomly sample in each step
    log_steps=10_000,  # How frequently should training progress be logged
    seed=SEED          # The RNG seed to use when sampling mini-batches (for reproducibility)
):
  if not learning_rate: learning_rate = LEARNING_RATE
  layers, parameters = model
  C = parameters[0]

  # Unpack model
  layers, parameters = model

  # Unpack dataset
  X, Y = dataset

  # same optimization as last time
  losses = []
  ud = []

  generator = torch.Generator().manual_seed(seed) # for reproducibility

  for step in range(n_steps):
    batch_indexes = torch.randint(0, X.shape[0], (batch_size,), generator=generator)
    Xbt, Ybt = X[batch_indexes].to(DEVICE), Y[batch_indexes].to(DEVICE)

    # forward pass
    Xemb = C[Xbt] # Embed characters into vectors
    Xembcat = Xemb.view(Xemb.shape[0], -1) # Concatenate the vectors # TODO: why not this: Xemb.view(-1, 6)
    x = Xembcat
    for layer in layers: x = layer(x)
    loss = F.cross_entropy(x, Ybt) # loss function

    # Perform backpropagation
    # TODO: this is required for debugging, why?
    for layer in layers: layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
    for p in parameters: p.grad = None # Reset gradients from previous step
    loss.backward() # Perform backward pass

    # update
    for p in parameters: p.data -= learning_rate * p.grad

    # Track stats
    if step % log_steps == 0 or step == n_steps - 1: print(f"{step+1:7d}/{n_steps:7d}: {loss.item():4f}")
    losses.append(loss.item())#loss.log10().item()) # TODO: why the log10?

    with torch.no_grad():
      ud.append([((learning_rate*p.grad).std() / p.data.std()).log10().item() for p in parameters]) # TODO: .log10().item()

  return losses, ud

model = build_model(vocabulary_size)
train(training_set, model, 1)

Now, let's recreate the `calculate_dataset_split_loss()` function:

In [ ]:
@torch.no_grad() # this decorator disables gradient tracking
def calculate_dataset_split_loss(
    model,
    split
):
  layers, parameters = model
  C = parameters[0]

  # put layers into eval mode
  original_training = [layer.training if hasattr(layer, "training") else None for layer in layers]
  for layer in layers: layer.training = False

  x,y = {
    'training': training_set,
    'validation': validation_set,
    'test': test_set,
  }[split]
  x = x.to(DEVICE)
  y = y.to(DEVICE)
  emb = C[x] # (N, block_size, n_embd)
  x = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  for layer in layers: x = layer(x)
  loss = F.cross_entropy(x, y)
  print(split, loss.item())

  original_training = [layer.training for layer in layers]
  for training in original_training:
    if training is not None: layer.training = training

calculate_dataset_split_loss(model, 'training')
calculate_dataset_split_loss(model, 'validation')

Finally, let's recreate the `sample_words()` function:

In [ ]:
def sample_words(
    model,
    num_words,
    block_size=BLOCK_SIZE,
    seed=SEED
):
    # Use generator for reproducibility
    generator = torch.Generator().manual_seed(seed)

    layers, parameters = model  # Unpack the model parameters
    C = parameters[0]

    # Pre-allocate context tensor (all zeros initially)
    context = torch.zeros(block_size, device=DEVICE, dtype=torch.long)  # Pre-create context on GPU

    # Sample N words
    words = []
    for _ in range(num_words):
        char_indexes = []
        context[:] = 0  # Reset context to all zeros at the start of each new word
        while True:
            # Embedding lookup (already on GPU)
            emb = C[context.unsqueeze(0)]  # (1, block_size, n_embd), no need to move to device
            x = emb.view(emb.shape[0], -1)  # Flatten embeddings

            # Forward pass through the layers
            for layer in layers:
                x = layer(x)

            logits = x  # Output from the model

            # Compute probabilities and sample the next character
            probs = F.softmax(logits, dim=1)  # Normalize logits to get probabilities
            char_i = torch.multinomial(probs, num_samples=1, generator=generator).item()  # Sample character

            # Update context (FIFO queue, pop oldest, add new char)
            context = torch.cat([context[1:], torch.tensor([char_i], device=DEVICE)])  # Update on GPU

            char_indexes.append(char_i)  # Add to list of sampled character indices

            # If EOS character sampled, break loop
            if char_i == EOS_CHAR_INDEX:
                break

        # Convert sampled character indexes to string using 'itos_map'
        word = "".join([itos_map[i] for i in char_indexes])
        words.append(word)  # Add word to list of sampled words

    return words

model = build_model(vocabulary_size)
result = train(training_set, model, 50_000)
calculate_dataset_split_loss(model, "training")
calculate_dataset_split_loss(model, "validation")
sample_words(model, 20)

## Inspect Model Behavior

First, let's write some code to plot various details about our model. You don't need to understand all of it in depth—just skim through to get an idea of what's happening. These are simply tools we'll use for the analysis that follows:

In [ ]:
import torch
import matplotlib.pyplot as plt
import math

def plot_layer_statistics(
    model,
    ax,
    logs,
    data_getter,
    layer_type=None,
    param_type=None,
    title="Statistics Distribution",
    subtitle=None,
    saturation_threshold=None,
    learning_rate=None,
    ud=None  # Add UD parameter here
):
    layers, parameters = model
    legends = []
    _logs = []

    if ud is not None:  # Check if UD data is provided
        for i, p in enumerate(parameters):
            if p.ndim == 2:  # Only consider 2D parameters
                ud_values = [ud[j][i] for j in range(len(ud))]
                ax.plot(ud_values)
                legends.append(f'param {i}')

        # Plot an indicator line for the ratio ~1e-3
        ax.plot([0, len(ud)], [-3, -3], 'k', label="~1e-3 ratio") # Ratios should be ~1e-3 on average

        # Set title and legends for the UD plot
        ax.set_title(title + (f"\n{subtitle}" if subtitle else ""))
        ax.legend(legends)

        return  # Exit after plotting UD data to avoid mixing with other types

    if layer_type:  # Working with layers (outputs or gradients)
        # Iterate through layers
        for i, layer in enumerate(layers[:-1]):
            # Skip layers that are not of the specified type
            if not isinstance(layer, layer_type): continue

            # Get the relevant data using the data_getter
            data = data_getter(layer)

            # Skip if data is None
            if data is None: continue

            # Add layer to legends
            legends.append(f"layer {i:2} ({layer_type.__name__})")

            # Compute statistics
            data_mean = data.mean()
            data_std = data.std()
            log_msg = f"layer {i:2} ({layer_type.__name__}): mean = {data_mean:8.6f} | std = {data_std:8.6f}"

            # Handle saturation for outputs if required
            if saturation_threshold is not None:
                saturation = (data.abs() > saturation_threshold).float().mean() * 100
                log_msg += f" | saturated = {saturation:6.2f}%"

            _logs.append(log_msg)

            # Create a histogram of the data
            hy, hx = torch.histogram(data, density=True)
            ax.plot(hx[:-1].detach(), hy.detach())

    else:  # Working with parameters (weights or gradients)
        # Iterate through all parameters
        for i, p in enumerate(parameters):
            # Filter by parameter type (weights only)
            if p.ndim == 2 and param_type == "weights":
                data = data_getter(p)
                if data is None: continue

                grad_mean = data.mean()
                grad_std = data.std()
                data_std = p.std()
                grad_data_ratio = grad_std * learning_rate / data_std

                _logs.append(f"weight {i} {tuple(p.shape)} | mean = {grad_mean:+.6f} | std = {grad_std:e} | update:data ratio = {grad_data_ratio:e}")

                # Create a histogram of the gradients
                hy, hx = torch.histogram(data, density=True)
                ax.plot(hx[:-1].detach(), hy.detach())

                legends.append(f'{i} {tuple(p.shape)}')

    # Set title and legends for the current subplot
    ax.set_title(title + (f"\n{subtitle}" if subtitle else ""))
    ax.legend(legends)

    log_header = title + f" ({subtitle})" if subtitle else ""
    logs.extend([f"##### {log_header}"] + _logs)

def plot_model_graphs(n_steps, layer_type=Tanh, non_linearity=Tanh, init_gain=5/3, batch_norm=None, learning_rate=LEARNING_RATE):
    model = build_model(vocabulary_size, non_linearity=non_linearity, kaiming_init_gain=init_gain, batch_norm=batch_norm)
    training_result = train(training_set, model, n_steps, learning_rate=learning_rate)

    num_plots = 4 if n_steps > 1 else 3
    num_cols = num_plots
    num_rows = math.ceil(num_plots / num_cols)
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(21, 4 * num_rows))
    axes = axes.flatten()

    # Call each plot function with the corresponding axis
    logs = []

    # TODO: change arg order
    # Plot the outputs
    plot_layer_statistics(
        model, axes[0], logs, lambda layer: layer.out, layer_type=layer_type, learning_rate=learning_rate,
        title="Layers Output Distribution", subtitle=f"init gain = {init_gain:.2f}" if init_gain else "", saturation_threshold=0.97
    )

    # Plot the gradients
    plot_layer_statistics(
        model, axes[1], logs, lambda layer: layer.out.grad, layer_type=layer_type, learning_rate=learning_rate,
        title="Layers Gradient Distribution", subtitle=f"init gain = {init_gain:.2f}" if init_gain else ""
    )

    # Plot the weights
    _, ud = training_result
    plot_layer_statistics(
        model, axes[2], logs, lambda param: param.grad, param_type="weights", learning_rate=learning_rate,
        title="Weights Gradient Distribution", subtitle=f"init gain = {init_gain:.2f}" if init_gain else ""
    )

    # Plot the update ratios
    if n_steps > 1:
      plot_layer_statistics(
          model, axes[3], logs,
          data_getter=lambda param: None,  # UD does not need to get data via data_getter, it's handled separately
          param_type="weights",  # We're dealing with weights in UD
          ud=ud,  # Pass UD data here
          title="UD Distribution",
          subtitle=f"init gain = {init_gain:.2f}" if init_gain else ""
      )

    # Adjust layout and show plot
    plt.tight_layout()
    plt.show()

    print("\n".join(logs))

Analyze tanh and its gradient:

In [ ]:
x = torch.linspace(-5, 5, 100)
y = torch.tanh(x)
plt.plot(x, y)

When $x < -2$ or $x > 2$, the $\tanh$ function starts flattening towards 0.0.

In [ ]:
for x in [-4, -2, 2, 4]:
  print(f"tanh({x}) = {torch.tanh(torch.tensor(x))}")

Now its gradient:

In [ ]:
x = torch.linspace(-5, 5, 100)
y = 1 - torch.pow(torch.tanh(x), 2)
plt.plot(x, y)

Here we confirm our interpretation. The gradient is at its max when X = 0, and at starts approach zero at [-2, 2] and then is pretty much zero when outside [-4, 4]:

In [ ]:
for x in [-4, -2, 2, 4]:
  print(f"grad(tanh({x})) = {1 - torch.pow(torch.tanh(torch.tensor(x)), 2)}")

## Tanh layers - no init

In [ ]:
plot_model_graphs(1, Tanh, init_gain=None) # TODO: set initialization type = None



This table now includes the specified values for "Tanh layers - no init" experiment. Let me know if you'd like to modify or add any other information!

Experiment name: Tanh layers - no init
Loss: 1
Stable Output: ✅
Uniform Act. Sat.: ✅
Acceptable Act. Sat: ❌
Stable Layer Gradients: ❌
Stable Weight Gradients: ❌
Optimal Update/Data Ratio: ❌
Uniform Update/Data Ratio: ❌

For this experiment `Tanh layers - no init`, after 1 training step, we can conclude the following:

**Outputs:**
- ❌ All tanh layers are very saturated. Without any weight initialization, weights are sampled from a standard normal distribution, meaning that their range will be [-3,3] most of the time. Meaning there is a high likelihood that weights will boost values beyond the range where tanh starts squeezing them. For this reason, we see high saturation across all layers. For one, by default, from the start, most inputs are being mostly squashed into ttwo values, -1 or 1, meaning we're losing signal in the forward pass.

**Tanh Layer Gradients:**
- ❌ Gradients become bigger as they backpropagate. This is because the values become smaller as they feed forward. Ideally we would have a similar gradient distribution across layers.

**Weights Gradients:**
- ❌ The same pattern is happening with the weight gradients, where they get bigger as they are backpropagated. Also, the gradients should be three orders of magnitude smaller than the data they're updating. Here they are too big, just one or two orders of magnitude smaller.

**Conclusion:**
- ❌ Bad starting state. The extreme tanh activations are a bad sign, if the model doesn't balance it out with more steps, learning will be quite slow.

Let's run longer:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=None) # TODO: set initialization type = None, TODO: replace 10000 -> 10_000

For this experiment `Tanh layers - no init`, after $10k$ training steps, we can conclude the following:

Here is the table formatted with the data you provided:

| Experiment   | Loss | Stable Output Prop. | Uniform Act. Sat. | Acceptable Act. Sat. | Stable Tanh Gradients | Stable Weight Gradients | Optimal Update/Data Ratio | Uniform Update/Data Ratio |
|--------------|------|---------------------|-------------------|----------------------|-----------------------|-------------------------|---------------------------|---------------------------|
| Tanh layers - no init | 2.59    | ✅                   | ✅                 | ❌                    | ❌                     | ❌                       | ❌                         | ❌                         |

**Outputs:**
- ❌ Tanh activations remain very saturated, curiously the first layer even worsened.

**Tanh Layer Gradients:**
- ❌ Gradients are still growing as they backpropagate.

**Weights Gradients:**
- ❌ Gradients are still growing as they backpropagate.

**Update/Data Ratio:**

- ❌ Layers are learning at very different rates, the magnitude of their updates in respect to their data is very high for some, and lower for others. The output layer is an outlier because it starts with an handicap where we artificially shrink the weights at initialization to force a uniform distribution in the output, which is the optimal starting state for the problem we're modeling. In a normal scenario, the model would start with big updates to sort out this handicap but start converging to a much smaller ratio with time, this isn't happening, which is indicative of the model not learning properly.

**Conclusion:**
- ❌ The model is failing to learn.

Let's log the result of this experiment:

| Experiment            | Loss |
|---------------------- |---------------|
| **Tanh layers - no init** | **2.88**    |

## Tanh layers - no init + batch norm

In [ ]:
plot_model_graphs(1, Tanh, init_gain=None, batch_norm=True) # TODO: set initialization type = None

For this experiment `Tanh layers - no init + batch norm`, after 1 training step, we can conclude the following:

**Outputs:**
- ✅ Before using batch normalizatino, all tanh layers were very saturated from the start. This is no longer the case, saturation starts low but evenly distributed across layers.

**Tanh Layer Gradients:**
- ✅ Beforehand, gradients were not stable, they grew as they backpropagated. They are now stable and similar all layers.

**Weights Gradients:**
- ❌ Weight gradient standard deviation is now stable during backpropagation.
Also, the ratio of gradients to the data they're updating is decent,but perhaps a bit too low.

**Conclusion:**
- ✅ Batch normalization seems to have mitigated the issues caused by not performing any model initialization.

Let's run longer:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=None, batch_norm=True) # TODO: set initialization type = None

For this experiment `Tanh layers - no init + batch norm`, after $10k$ training steps, we can conclude the following:

**Outputs:**
- ✅ Tanh saturations remain consistent with first training step, low and consistent across layers.

**Tanh Layer Gradients:**
- ✅ Gradients remain stable during backpropagation, with a minor increase in value as they get backpropagated. Their standard deviation has increased by an order of magnitude when compared to first step.

**Weights Gradients:**
- ✅ Like in the tahn layers, gradients exhibit the same pattern, stable across layers, slighly increasing as they move to the first layers. Their standard deviation has also increased by an order of magnitude when compared to the first step, the gradient to data ratio as also increased by an order of magnitude making it more indicative of the model picking up the learning pace.

**Update/Data Ratio:**

- ❌ All layers except the first one are learning at a slow rate. As can be seen in the *UD Distribution* graph, the ratio of updates to the data being updated is below the optimal of 1**-3 for most layers.

**Conclusion:**
- ❌ The mitigation that happened at initializtaion has lasted through training with training being very stable and resulting in a much better loss than not having used batch normalization. However, the model is learning slower than it could.

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| **Tanh layers - no initialization + batch normalization** | **2.11**    |

Let's try increasing the learning rate to fix the issue of the layers learning slowly:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=None, batch_norm=True, learning_rate=LEARNING_RATE*10)

TODO: WHY THE WORSE LOSS?

With a 10x learning rate we got a better loss. There are some noticeable patterns that justify it:

**Outputs:**
- Output standard deviation has increased resulting in higher tanh saturation, yet these patterns are stable across all layers. It seems that the model has learned faster to leverage the non-linearity in order to produce a lower loss.

**UD Distribution**:
- All layers are now learning at a good rate, with the initial layer performing updates with the highest magnitude.

Due to this conclusion, whenever we try batch norm going forward, we are going to 10x the learning rate, unless we need to adjust it again if we notice an issue in the UD distribuution.

Let's log the results:

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| Tanh layers - no init + batch norm | 2.11    |
| **Tanh layers - no init + batch norm + 10x learning rate** | **2.21**    |


## Tanh layers - optimal init gain

Let's analyze the model when we use the proper init gain for tanh of 5/3:

In [ ]:
plot_model_graphs(1, Tanh, init_gain=5/3)

For this experiment `Tanh layers - optimal init gain`, after 1 training step, we can conclude the following:

**Outputs:**
- ✅ Well-distributed values across the $tanh$ range of $[-1, 1]$.
- ✅ Stable standard deviation across layers, ensuring smooth signal propagation.
- ✅ Healthy $tanh$ saturation: values avoid gradient dead zones, with controlled saturation allowing non-linear expressivity.
- ✅ Slightly higher saturation in the first layer, decreasing as $tanh$ compresses values into $[-1, 1]$. Strong weights still push some values to extremes, though limited.

**Tanh Layer Gradients:**
- ✅ Similar gradient distribution across layers, ensuring effective backpropagation and a solid start for learning.

**Weights Gradients:**
- ❌ Gradients smaller than weights, except in the last layer where they match in scale, this is caused by the output layer weight squeeze.
- ❌ Output layer gradients are about $100x$ larger than earlier layers.

**Conclusion:**
- ✅ Good initial activation and gradient flow.
- ❌ Large weights and gradient-to-weight ratio in the last layer suggest possible instability, needing further analysis.

Let's see if the issue we detected in the last layer goes away with further training:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=5/3)

For this experiment `Tanh layers - optimal init gain`, after $10k$ training steps, we can conclude the following:

**Outputs:**
- ✅ Non-linearity now appears across all layers. Initially, only the first layer showed $\tanh$ saturation, but now all layers exhibit mild saturation, indicating a stronger usage of its non-linear properties.

**Tanh Layer Gradients:**
- ✅ Gradient distribution remains consistent across layers, suggesting uniform learning speed across layers.
- ✅ Gradient standard deviation has increased, reflecting dynamic weight adjustments during training.

**Weight Gradients:**
- ✅ The earlier weight discrepancy in the last layer has significantly reduced, showing the model is correcting the output layer imbalance.

**Update/Data Ratios:**
- ✅ Most layers update with magnitudes $1000$ times smaller than their parameter values, a healthy ratio.
- ✅ Output layer update ratio starts large but shrinks as initial weight capping is resolved. -- TODO

**Conclusion:**
- ✅ The model is leveraging non-linearities more effectively.
- ✅ Gradient flow remains stable.
- ✅ The weight imbalance in the output layer is resolving.
- ✅ Overall, the training process is going well as the model is learning effectively.

Let's log the results:

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| Tanh layers - no init + batch norm | 2.11    |
| Tanh layers - no init + batch norm + 10x learning rate | 2.21    |
| **Tanh layers - optimal init gain** | **1.99**    |


## Tanh layers - optimal init gain + batch norm

In [ ]:
plot_model_graphs(1, Tanh, init_gain=5/3, batch_norm=True)

For this experiment `Tanh layers - optimal init gain + batch norm`, after $1$ training step, we can conclude the following:

**Outputs:**
- ✅ Stable tanh saturation, altough a bit low.

**Tanh Layer Gradients:**
- BAD - Stable gradients, although low and growing as they get backpropagated.


**Weight Gradients:**
- ✅ Stable gradients, growing as they get backrpopagated. Gradients are probably too big for the data they're updating, but not too bad.

**Conclusion:**
- ✅ Model seems stable at initialization and ready to learn.

Train longer:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=5/3, batch_norm=True)

The update to data ratios are decent, but from the UD graph its noticeable that the first layer is learning optimally while the others are learning at a much slower rate in comparison, lets just nudge the learning rate a bit to make them all be closer to the baseline:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=5/3, batch_norm=True, learning_rate=LEARNING_RATE * 0.5)

For this experiment `Tanh layers - optimal init gain + batch norm`, after $10k$ training steps, we can conclude the following:

**Outputs:**
- ✅ Stable saturation across layers.

**Tanh Layer Gradients:**
- ✅ Stable gradients across layers, growing as they get backpropagated, but within a healthy standard deviation.

**Weight Gradients:**
- ✅ Stable gradients, growing as they get backrpopagated. Updates may be a bit too big for the values they're updating.

**Conclusion:**
- ✅ Training is stable and model is learning well. The loss was worse than without batch norm however, but it could improve with a lower learning rate.

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| Tanh layers - no init + batch norm | 2.11    |
| Tanh layers - no init + batch norm + 10x learning rate | 2.21    |
| Tanh layers - optimal init gain | 1.99    |
| **Tanh layers - optimal init gain + batch norm** | **1.96**    |

## Tanh layers - small init gain

Now let's analyze it when we use a small init gain of $1/2$ for 1 step:

In [ ]:
plot_model_graphs(1, Tanh, init_gain=1/2)

For this experiment `Tanh layers - small init gain`, after $1$ training step, we can conclude the following:

**Outputs:**
- ❌ Outputs shrink as they propagate forward through the layers, as $\tanh$ squeezes them into $[-1, 1]$, and the small gain further amplifies this shrinking effect.

**Tanh Layer Gradients:**
- ❌ Gradients are extremely small, with a standard deviation around $1e^{-4}$ (anything smaller than $1e^{-2}$ starts being too small).
- ❌ Vanishing gradients: Gradients diminish layer by layer during backpropagation, eventually reaching values as small as $1e^{-5}$ when they reach the first layer.

**Weights:**
- ❌ Last layer weight gradients are about $10x$ larger than in earlier layers.
**Conclusion:**
- ❌ The network will struggle to learn, as vanishing gradients impede backpropagation.
- ❌ Severe compression of values limits $\tanh$ activation, blocking necessary non-linear expressivity for learning.

Now let's train it longer:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=1/2)

For this experiment `Tanh layers - small init gain`, after $10k$ training steps, we can conclude the following:

**Outputs**:
- Tanh saturation increased slightly

**Tanh Layer Gradients:**
- Gradient magnitudes have increased as well and stabilizied across layers.

**Weights Gradients:**
- Gradient weight distribution also stabilized.

**Update Data Ratio**:
- All layers are converging to a similar learning speed except the first layer which is struggling, this is caused by the vanishing gradient problem.

**Conclusion**:
- The small init gain is hampering the ability of the model to learn effectively.

Let's log the results:

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| Tanh layers - no init + batch norm | 2.11    |
| Tanh layers - no init + batch norm + 10x learning rate | 2.21    |
| Tanh layers - optimal init gain | 1.99    |
| **Tanh layers - optimal init gain + batch norm** | **1.96**    |
| Tanh layers - small init gain | 2.14    |

## Tanh layers - small init gain + batch norm

In [ ]:
plot_model_graphs(1, Tanh, init_gain=1/2, batch_norm=True)

For this experiment `Tanh layers - small init gain + batch norm`, after $1$ training step, we can conclude the following:

**Outputs**:
- Tanh saturation is stable across layers.

**Tanh Layer Gradients:**
- Gradients are stable across layers.

**Weights Gradients:**
- Gradients are stable across layers.

**Update Data Ratio**:
- Ratios are a bit too high.

**Conclusion**:
- Model is stable and should learn effectively.

Now longer:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=1/2, batch_norm=True)

There is a big discraMost layers are learning too fast, by lowering the learning rate we should get better results:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=1/2, batch_norm=True, learning_rate=LEARNING_RATE * 10**-1)

For this experiment `Tanh layers - small init gain + batch norm`, after $10k$ training steps, we can conclude the following:

**Outputs**:
- Tanh saturation is stable across layers. Its increasing as model is learning to express its non-linearity.

**Tanh Layer Gradients:**
- Gradients are stable across layers and increased by an order of magnitude.

**Weights Gradients:**
- Gradients are stable across layers and increased by an order of magnitude.

**Update Data Ratio**:
- Ratios have a good magnitude and are stable across layers (except for first layer where ratio is an order of magnitude lower)

**Conclusion**:
- Model was able to overcome its bad initialization and got a decent loss value, not as good as with propert initialization as well.

Let's log the results:

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| Tanh layers - no init + batch norm | 2.11    |
| Tanh layers - no init + batch norm + 10x learning rate | 2.21    |
| Tanh layers - optimal init gain | 1.99    |
| **Tanh layers - optimal init gain + batch norm** | **1.96**    |
| Tanh layers - small init gain | 2.14    |
| Tanh layers - small init gain + batch norm | 2.02  |

## Tanh layers - no init gain

Now let's analyze the $tanh$ layer flow at initialization when the gain is $1$ (no gain):

In [ ]:
plot_model_graphs(1, Tanh, init_gain=1)

For this experiment `Tanh layers - no init gain`, after $1$ training step, we can conclude the following:

**Outputs**:
- Tanh saturation degrades to zero across layers.

**Tanh Layer Gradients:**
- Gradients are stable across layers slightly decreasing as they backpropagate.

**Weights Gradients:**
- Weight gradients are stable across layers although a bit small. The output layer's weights are 100x bigger than remaining layers though, and its updates are as big as their data.

**Conclusion:**
- The model will probably be able to learn fine as it can use its weights to compensate for the lack of post activation gain. Its just not starting from the best state.

And now train it longer:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=1)

For this experiment `Tanh layers - no init gain`, after $10k$ training steps, we can conclude the following:

**Outputs**:
- Tanh saturation is improving across layers as epxpected.

**Tanh Layer Gradients:**
- Gradients are stable across layers.

**Weights Gradients:**
- Gradients are stable across layers with the last layer still lagging behind.

**Conclusion:**
- The model performed good enough.

Let's log the results:

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| Tanh layers - no init + batch norm | 2.11    |
| Tanh layers - no init + batch norm + 10x learning rate | 2.21    |
| Tanh layers - optimal init gain | 1.99    |
| **Tanh layers - optimal init gain + batch norm** | **1.96**    |
| Tanh layers - small init gain | 2.14    |
| Tanh layers - small init gain + batch norm | 2.02  |
| Tanh layers - no init gain | 2.11  |

## Tanh layers - no init gain + batch norm

In [ ]:
plot_model_graphs(1, Tanh, init_gain=1, batch_norm=True)

For this experiment `Tanh layers - no init gain + batch norm`, after 1 training step, we can conclude the following:

**Outputs**:
- Tanh saturation is stable across layers.

**Tanh Layer Gradients:**
- Gradients are stable across layers.

**Weights Gradients:**
- Gradients are stable across layers.

**Conclusion:**
- The model is stable and ready to learn.

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=1, batch_norm=True)

Again we may get better loss from lower learning rate:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=1, batch_norm=True, learning_rate=LEARNING_RATE*10**-1)

For this experiment `Tanh layers - no init gain + batch norm`, after $10k$ training steps, we can conclude the following:

**Outputs**:
- Tanh saturation is stable across layers.

**Tanh Layer Gradients:**
- Gradients are stable across layers.

**Weights Gradients:**
- Gradients are stable across layers.

**Update data ratios:**
- Ratios are stable across layers.

**Conclusion:**
- The model is stable and ready to learn.

Let's log the results:

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| Tanh layers - no init + batch norm | 2.11    |
| Tanh layers - no init + batch norm + 10x learning rate | 2.21    |
| Tanh layers - optimal init gain | 1.99    |
| **Tanh layers - optimal init gain + batch norm** | **1.96**    |
| Tanh layers - small init gain | 2.14    |
| Tanh layers - small init gain + batch norm | 2.02  |
| Tanh layers - no init gain | 2.11  |
| Tanh layers - no init gain + batch norm | 2.05 |

## Tanh layers - large init gain

And now when the init gain is large:

In [ ]:
plot_model_graphs(1, Tanh, init_gain=3)

**Outputs**:
- With a init gain of 3, tanh outputs are mostly -1 and 1. The gain is consistently moving the value to the extremes of the range where tanh starts squeezing values to -1 and 1. After each of these comprissions, the gain pushes the value outwards beyond the range [-2, 2] again, making the next tanh operation compress it back to -1 or 1 once more.

**Tanh Layer Gradients:**
- Most gradients are very close to zero, with the ones in the earlier layers being the biggest, since their values are large enough comparing to other layers as to compensate for the vanishing gradient caused by the chain rule.

**Weights Gradients:**
- Weight gradients are small.

**Conclusion:**
- This network should not learn at all, even if the first layer learns something, the last layers cant adapt any of their parameters so ultimately no classification is learned.

Let's train for longer:

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=3)

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| **Tanh layers - gain = 5/3** | **2.01**    |
| Tanh layers - gain = 1/2 | 2.24    |
| Tanh layers - gain = 1   | 2.02    |
| Tanh layers - gain = 3   | 1.99    |

**Outputs**:
- Output distribution is consistent with initialization.

**Tanh Layer Gradients:**
- Gradients are small
- Gradients become smaller as they backpropagate from last layer to first (vanishing gradient problem).

**Weights Gradients:**
- Gradients are small
- Gradients become smaller as they backpropagate from last layer to first (vanishing gradient problem).

**Update data ratio:*
- All seem ok, but output layer doesnt seem to be fixing itself.

**Conclusion:**
- Network is learning something.

## Tanh layers - large init gain + batch norm

In [ ]:
plot_model_graphs(1, Tanh, init_gain=3, batch_norm=True)

Run the following analysis.

In [ ]:
plot_model_graphs(10_000, Tanh, init_gain=3, batch_norm=True)



---

## Linear model - optimal tanh gain

Now, let's examine what happens to the forward and backward pass when we remove the $tanh$ non-linearities:

In [ ]:
plot_model_graphs(1, Linear, non_linearity=None, init_gain=5/3)

**Outputs**:
- Value magnitude increases across layers, starting with a std of 2 and ending with 15.

**Linear Layer Gradients:**
- Gradients are very small.
- Gradients increase as they backpropagate.

**Weights Gradients:**
- Gradients are very small.
- Last layer issue.

**Conclusion:**
- Network will struggle to learn.

Let's train it longer:

In [ ]:
plot_model_graphs(4, Linear, non_linearity=None, init_gain=5/3) # TODO: change to log_freq=1

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| **Tanh layers - gain = 5/3** | **2.01**    |
| Tanh layers - gain = 1/2 | 2.24    |
| Tanh layers - gain = 1   | 2.02    |
| Linear model - gain = 5/3   | 572    |

**Outputs**:
- Values start increasing exponentially as they propagate through layers.

**Linear Layer Gradients:**
- Exploding gradients: Gradients start increasing exponentially as they propagate through layers.

**Weights Gradients:**
- Exploding gradients: Gradients start increasing exponentially as they propagate through layers.

**Update ratio:**
- Updates are orders of magnitude higher than the data they are updating.

**Conclusion:**
- Model is fully unable to learn, exploding gradients are leading to huge losses that worsen with training.

## Linear model - optimal tanh gain + batch norm

In [ ]:
plot_model_graphs(1, Linear, non_linearity=None, init_gain=5/3, batch_norm=True)

Run the following analysis.

In [ ]:
plot_model_graphs(10_000, Linear, non_linearity=None, init_gain=5/3, batch_norm=True)


## Linear model - optimal linear gain

In [ ]:
plot_model_graphs(1, Linear, non_linearity=None, init_gain=1)

**Outputs**:
- Values propagation is stable with values assuming same range across all layers.

**Linear Layer Gradients:**
- Gradients are small but stable across layhers.

**Weights Gradients:**
- Gradients are small but stable across layhers. Except last.

**Conclusion:**
- Model should be able to learn, but should have worse loss than one with a non-linear layer.

Let's train longer:

In [ ]:
plot_model_graphs(10_000, Linear, non_linearity=None, init_gain=1)

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| **Tanh layers - gain = 5/3** | **2.01**    |
| Tanh layers - gain = 1/2 | 2.24    |
| Tanh layers - gain = 1   | 2.02    |
| Linear model - gain = 5/3   | 572    |
| Linear model - gain = 1   | 2.32    |

**Outputs**:
- Values are smaller but remaing stable across layers. The trend is for values to become smaller as they propagate through layers.

**Linear Layer Gradients:**
- Gradients are small but stable across layers.

**Weights Gradients:**
- Gradients are small but stable across layers. Output layer is improving.

**Update data ratio:**
- Most layers are learning at a healthy rate.
- First layer is struggling to learn comparing to others, but not too bad.
- Output layer update magnitude lowers with each step.

**Conclusion:**
- Model is learning.

## Linear model - optimal linear gain + batch norm

In [ ]:
plot_model_graphs(1, Linear, non_linearity=None, init_gain=1)

Run the following analysis.

In [ ]:
plot_model_graphs(10_000, Linear, non_linearity=None, init_gain=1, batch_norm=True)


## Linear model - small init gain

In [ ]:
plot_model_graphs(1, Linear, non_linearity=None, init_gain=1/2)

**Outputs**:
- Values shrink as they move across layers.

**Linear Layer Gradients:**
- Vanishing gradient problem: Gradients are small and drop to zero as they backpropagate.

**Weights Gradients:**
- Vanishing gradient problem: Gradients are small and drop to zero as they backpropagate.
- Issue with output layer.

**Conclusion:**
- Model will struggle to learn because gradients are failing to backpropagate.

Train for longer:

In [ ]:
plot_model_graphs(10_000, Linear, non_linearity=None, init_gain=1/2)

| Experiment            | Loss |
|---------------------- |---------------|
| Tanh layers - no initialization | 2.72    |
| **Tanh layers - gain = 5/3** | **2.01**    |
| Tanh layers - gain = 1/2 | 2.24    |
| Tanh layers - gain = 1   | 2.02    |
| Linear model - gain = 5/3   | 572    |
| Linear model - gain = 1   | 2.32    |
| Linear model - gain = 1/2   | 2.34    |

**Outputs**:
- Values propagation has improved.

**Linear Layer Gradients:**
- Gradient propagation has improved.

**Weights Gradients:**
- Gradient propagation has improved.

**Data update ratio:**
- Update rates are improving and converging across layers.

**Conclusion:**
- Model has been able to overcome bad initialization and found a way to backpropagate gradients and is now learning.


## Linear model - small init gain + batch norm

In [ ]:
plot_model_graphs(1, Linear, non_linearity=None, init_gain=1/2, batch_norm=True)

Run the following analysis.

In [ ]:
plot_model_graphs(10_000, Linear, non_linearity=None, init_gain=1/2, batch_norm=True)

## Final Run

This is our final benchmark, lets pick up the leading candidate without batch norm and compare it against it with batch norm on a long run.

## Conclusions

Amongst all the previous experiments the following patterns were common when we fiddled with different params:

TODO